> **Versión de pruebas 0.2.0:** esta interfaz está en la PR #4; el enlace original de main todavía no se ha actualizado.

# MEIKA Vox · transcribir audios desde Google Drive

**Gratis y de código abierto. No necesitas programar.**

1. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, selecciona **GPU** si está disponible.
2. Pulsa **Entorno de ejecución → Ejecutar todas**. Espera la preparación y autoriza tu Drive.
3. En el panel final, elige proyecto y carpeta, selecciona audios y pulsa **Transcribir seleccionados**.

Los textos se guardan en tu Drive y puedes descargarlos como ZIP desde el panel.
**Distinguir hablantes es opcional:** requiere una cuenta gratuita y clave de Hugging Face; el panel explica cómo activarla.

Colab procesa tus grabaciones en infraestructura de Google y el código autorizado puede acceder a tu Drive. Los audios no se publican en GitHub. Colab gratuito tiene GPU y duración variables. Las transcripciones requieren revisión humana.


## Preparar herramienta
Ejecuta todo una vez. El código está plegado; no necesitas editarlo.


In [ ]:
import importlib.util
import subprocess
import sys
import threading
from pathlib import Path

if any(t.name == "MEIKA_Vox_batch" and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError("Hay un lote activo. Deténlo desde el panel o espera a que termine.")

REPO = Path("/content/meika-vox")
# La interfaz y el motor se descargan de la misma versión pública.
REPO_REF = "feat/colab-simple-user-mode-20261009"


def checked(command):
    result = subprocess.run([str(item) for item in command], capture_output=True, text=True)
    if result.returncode:
        print(result.stderr[-3000:] or result.stdout[-3000:])
        raise RuntimeError(
            "La preparación falló. Comprueba la conexión y ejecuta esta celda de nuevo."
        )


print("1/3 · Preparando FFmpeg y Git…")
checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])
print("2/3 · Descargando MEIKA Vox…")
if not (REPO / ".git").exists():
    checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
print("3/3 · Instalando dependencias; la primera preparación puede tardar varios minutos…")
checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "-e", str(REPO) + "[whisperx]", "ipywidgets>=8,<9"])
# Si se reejecuta el cuaderno en la misma sesión, no conservar módulos de la versión anterior.
for name in list(sys.modules):
    if name == "meika_vox" or name.startswith("meika_vox."):
        del sys.modules[name]
sys.path.insert(0, str(REPO / "src"))
runtime = importlib.import_module("meika_vox.runtime").inspect_runtime()
if not runtime.asr_ready:
    raise RuntimeError(
        "La instalación quedó incompleta. Reinicia la sesión y ejecuta todo de nuevo."
    )
print("Preparación comprobada. Ahora autoriza tu Google Drive.")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive conectado. Preparando panel…")


## Panel de transcripción
Selecciona una carpeta concreta dentro de Mi unidad.


In [ ]:
from google.colab import output

output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location("vox_ui", REPO / "scripts/colab_ui.py")
ui = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ui)
ui.build_panel(REPO, Path("/content/drive/MyDrive"))
